# ⚛️ Módulo 5: Dinámica Molecular
## Actividad 5.5: Preparación de Sistemas – Solvatación e Iones

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_05_dinamica_molecular/05_preparacion_sistemas.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Preparar estructuras moleculares para simulación de DM con GROMACS y OpenMM
- Limpiar estructuras de proteínas (remover ligandos, añadir hidrógenos, reparar cadenas)
- Solvatatar sistemas moleculares con modelos de agua explícita (TIP3P, SPC/E)
- Neutralizar y ajustar la concentración salina del sistema
- Generar topologías moleculares con GROMACS
- Construir cajas de simulación con la distancia mínima correcta al soluto

---

## 1. Instalación de Dependencias

In [ ]:
!pip install biopython requests numpy matplotlib py3Dmol
# GROMACS se instala con: conda install -c conda-forge gromacs
# OpenMM:                conda install -c conda-forge openmm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import requests
from pathlib import Path
from Bio import PDB

print("Bibliotecas importadas correctamente")

## 2. Flujo de Preparación de un Sistema para DM

```
Estructura PDB cruda
       ↓
1. Limpieza y reparación
   (remover agua, heteroátomos, reparar cadenas)
       ↓
2. Protonación
   (añadir H, verificar estados de protonación a pH 7)
       ↓
3. Generar topología
   (campos de fuerza: AMBER, CHARMM, GROMOS)
       ↓
4. Definir caja de simulación
   (cúbica, dodecaedro, distancia al borde ≥ 1 nm)
       ↓
5. Solvatación
   (añadir moléculas de agua explícita: TIP3P, SPC/E)
       ↓
6. Neutralización + iones
   (Na⁺/Cl⁻ para neutralizar y ajustar fuerza iónica)
       ↓
Sistema listo para minimización y simulación
```

## 3. Descarga y Limpieza de Estructura PDB

In [ ]:
def descargar_pdb(pdb_id, output_dir="estructuras_dm"):
    """
    Descarga un archivo PDB desde RCSB.
    
    Args:
        pdb_id: código PDB de 4 caracteres
        output_dir: directorio de salida
    
    Returns:
        Path al archivo descargado
    """
    Path(output_dir).mkdir(exist_ok=True)
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    r = requests.get(url)
    if r.status_code == 200:
        p = Path(output_dir) / f"{pdb_id}.pdb"
        p.write_text(r.text)
        print(f"✓ Descargado: {p}")
        return p
    else:
        print(f"✗ Error: {r.status_code}")
        return None

def limpiar_pdb(entrada, salida, remover_heteroatoms=True,
                remover_agua=True, cadenas=None):
    """
    Limpia un archivo PDB para simulación de DM.
    
    Args:
        entrada: ruta al PDB crudo
        salida: ruta al PDB limpio
        remover_heteroatoms: si remover HETATM (ligandos, cofactores)
        remover_agua: si remover moléculas de agua cristalográfica
        cadenas: lista de cadenas a conservar (None = todas)
    """
    parser = PDB.PDBParser(QUIET=True)
    estructura = parser.get_structure("proteina", entrada)
    
    class CleanSelect(PDB.Select):
        def accept_residue(self, residue):
            # Excluir agua
            if remover_agua and residue.resname in ('HOH', 'WAT', 'TIP3'):
                return False
            # Excluir heteroátomos (ligandos)
            if remover_heteroatoms and residue.id[0] != ' ':
                return False
            return True
        
        def accept_chain(self, chain):
            if cadenas is not None:
                return chain.id in cadenas
            return True

    io = PDB.PDBIO()
    io.set_structure(estructura)
    io.save(str(salida), CleanSelect())
    print(f"✓ PDB limpio guardado en: {salida}")

# Descargar y limpiar lisozima
pdb_raw = descargar_pdb("1AKI")
if pdb_raw:
    limpiar_pdb(pdb_raw, "estructuras_dm/1AKI_limpio.pdb")

## 4. Modelos de Agua

El modelo de agua es crucial para la calidad de la simulación. Los más utilizados son:

| Modelo | Tipo | Sitios | Uso recomendado |
|--------|------|--------|----------------|
| **TIP3P** | Rígido | 3 | AMBER, CHARMM, rápido |
| **SPC/E** | Rígido | 3 | GROMOS, propiedades de transporte |
| **TIP4P/2005** | Rígido | 4 | Mejores propiedades termodinámicas |
| **TIP5P** | Rígido | 5 | Mejor para agua líquida a 298 K |
| **OPC** | Rígido | 4 | Muy preciso, recomendado con AMBER |

> Para biomoléculas en general se usa **TIP3P** (AMBER/CHARMM) o **SPC/E** (GROMOS/GROMACS).

In [ ]:
# Comparar propiedades de modelos de agua
datos_agua = {
    'Modelo':           ['TIP3P', 'SPC/E', 'TIP4P/2005', 'TIP5P', 'OPC', 'Experimental'],
    'Densidad (g/cc)':  [0.982,   0.997,   0.999,        0.999,   0.997, 0.997],
    'Dipolo (D)':       [2.35,    2.35,    2.31,          2.29,   2.48,  2.95],
    'Tmelt (K)':        [146,     215,     252,            274,    249,   273],
    'Tmax_dens (K)':    [None,    241,     278,            277,    277,   277],
    'D (10⁻⁹ m²/s)':   [5.19,    2.49,    2.08,           2.62,   2.30,  2.30],
}

import pandas as pd
df_agua = pd.DataFrame(datos_agua)
print("Comparación de Modelos de Agua")
print("=" * 70)
print(df_agua.to_string(index=False))
print()
print("D = coeficiente de difusión (autódifusión del agua a 298 K)")

## 5. Solvatación y Adición de Iones con GROMACS

In [ ]:
# Flujo de preparación con GROMACS
flujo_gromacs = """
# ============================================================
# PREPARACIÓN COMPLETA CON GROMACS
# ============================================================
# Proteína de ejemplo: Lisozima de huevo de gallina (1AKI)

# 1. Generar topología con campo de fuerza AMBER99SB-ILDN
gmx pdb2gmx \\
    -f 1AKI_limpio.pdb \\
    -o 1AKI_processed.gro \\
    -water spce \\
    -ff amber99sb-ildn
# Genera: topol.top, posre.itp, 1AKI_processed.gro

# 2. Definir caja de simulación (dodecaedro rómbico, borde 1.0 nm)
gmx editconf \\
    -f 1AKI_processed.gro \\
    -o 1AKI_box.gro \\
    -c \\
    -d 1.0 \\
    -bt dodecahedron

# 3. Solvatación con agua SPC/E
gmx solvate \\
    -cp 1AKI_box.gro \\
    -cs spc216.gro \\
    -o 1AKI_solv.gro \\
    -p topol.top
# Añade automáticamente SOL al topol.top

# 4. Preparar para añadir iones
gmx grompp \\
    -f ions.mdp \\
    -c 1AKI_solv.gro \\
    -p topol.top \\
    -o ions.tpr

# 5. Añadir iones Na+ y Cl- (neutralizar + 0.15 M NaCl)
gmx genion \\
    -s ions.tpr \\
    -o 1AKI_solv_ions.gro \\
    -p topol.top \\
    -pname NA \\
    -nname CL \\
    -neutral \\
    -conc 0.15
# Seleccionar: SOL (grupo de agua para reemplazar con iones)
"""
print(flujo_gromacs)

In [ ]:
# Visualizar esquema de la caja de simulación
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# Caja de simulación
caja = patches.Rectangle((0.5, 0.5), 9, 9, linewidth=2,
                           edgecolor='black', facecolor='lightcyan', alpha=0.5)
ax.add_patch(caja)

# Proteína
proteina = patches.Ellipse((5, 5), 3.5, 3.0, linewidth=2,
                             edgecolor='steelblue', facecolor='steelblue', alpha=0.7)
ax.add_patch(proteina)

# Moléculas de agua (puntos)
np.random.seed(42)
agua_x, agua_y = [], []
for _ in range(200):
    x, y = np.random.uniform(0.7, 9.3), np.random.uniform(0.7, 9.3)
    # Excluir zona de la proteína
    if ((x-5)/1.75)**2 + ((y-5)/1.5)**2 > 1.1:
        agua_x.append(x)
        agua_y.append(y)
ax.scatter(agua_x, agua_y, c='royalblue', s=8, alpha=0.5, zorder=3, label='Agua')

# Iones
ion_x = np.random.uniform(1, 9, 6)
ion_y = np.random.uniform(1, 9, 6)
ax.scatter(ion_x[:3], ion_y[:3], c='orange', s=60, marker='^', zorder=4, label='Na⁺')
ax.scatter(ion_x[3:], ion_y[3:], c='green',  s=60, marker='v', zorder=4, label='Cl⁻')

# Flecha de distancia mínima
ax.annotate('', xy=(0.5, 5), xytext=(3.25, 5),
            arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax.text(1.3, 5.3, '≥ 1 nm', color='red', fontsize=11, fontweight='bold')

ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.set_title('Sistema Proteína en Solución Acuosa\n(caja de simulación con PBC)', fontsize=13)
ax.text(5.0, 5.0, 'Proteína', ha='center', va='center', color='white',
        fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig('sistema_solvatado.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Estimación del Número de Moléculas de Agua

In [ ]:
def estimar_agua_solvatacion(n_residuos, d_borde=1.0, tipo_caja='cubica',
                              densidad_agua=33.3):
    """
    Estima el número de moléculas de agua necesarias para solvatación.
    
    Args:
        n_residuos: número de residuos de aminoácidos
        d_borde: distancia mínima proteína-borde de caja (nm)
        tipo_caja: 'cubica' o 'dodecaedro'
        densidad_agua: moléculas de agua por nm³ (agua a 300 K ≈ 33.3)
    
    Returns:
        Estimación de número de moléculas de agua
    """
    # Radio esférico estimado de la proteína
    # Relación empírica: r_proteina ≈ 0.12 * N_residuos^(1/3) nm
    r_proteina = 0.12 * n_residuos**(1/3)  # nm
    
    # Volumen de la proteína
    V_proteina = (4/3) * np.pi * r_proteina**3  # nm³
    
    # Dimensión de la caja
    L = 2 * (r_proteina + d_borde)  # nm
    
    if tipo_caja == 'cubica':
        V_caja = L**3
    elif tipo_caja == 'dodecaedro':
        # Volumen dodecaedro rómbico ≈ 0.707 * L³
        V_caja = 0.707 * L**3
    
    V_agua = V_caja - V_proteina
    n_agua = int(V_agua * densidad_agua)
    n_atomos_total = n_residuos * 15 + n_agua * 3  # aprox.
    
    return {
        'r_proteina_nm': r_proteina,
        'L_caja_nm': L,
        'V_caja_nm3': V_caja,
        'n_agua': n_agua,
        'n_atomos_total': n_atomos_total
    }

# Proteínas de ejemplo
proteinas = [
    ('Lisozima', 129),
    ('Ubiquitina', 76),
    ('DHFR', 186),
    ('T4 Lisozima', 164),
    ('Proteína de prueba', 300),
]

print(f"{'Proteína':<20} {'Nres':>6} {'r(nm)':>7} {'L(nm)':>7} {'N agua':>8} {'N átomos':>10}")
print("-" * 63)
for nombre, nres in proteinas:
    for tipo in ['cubica', 'dodecaedro']:
        r = estimar_agua_solvatacion(nres, tipo_caja=tipo)
        print(f"{nombre:<20} {nres:>6} {r['r_proteina_nm']:>7.2f} "
              f"{r['L_caja_nm']:>7.2f} {r['n_agua']:>8,} "
              f"{r['n_atomos_total']:>10,}  ({tipo})")

## 7. Preparación con OpenMM

In [ ]:
# Preparación completa con OpenMM
codigo_openmm = """
# Preparación y simulación con OpenMM
import openmm as mm
import openmm.app as app
import openmm.unit as unit

# 1. Cargar PDB limpio
pdb = app.PDBFile('1AKI_limpio.pdb')

# 2. Seleccionar campo de fuerza
forcefield = app.ForceField(
    'amber14-all.xml',      # proteínas
    'amber14/tip3pfb.xml'   # agua TIP3P-FB
)

# 3. Construir sistema con solvatación automática
modeller = app.Modeller(pdb.topology, pdb.positions)
modeller.addHydrogens(forcefield, pH=7.0)

# Añadir agua y iones en un solo paso
modeller.addSolvent(
    forcefield,
    model='tip3p',
    padding=1.0*unit.nanometer,   # distancia mínima proteína-borde
    ionicStrength=0.15*unit.molar  # 0.15 M NaCl
)

print(f"Átomos totales: {modeller.topology.getNumAtoms()}")
print(f"Residuos totales: {modeller.topology.getNumResidues()}")

# 4. Crear sistema físico
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.0*unit.nanometer,
    constraints=app.HBonds,
    hydrogenMass=1.5*unit.amu  # masa aumentada para pasos de 4 fs
)

# 5. Guardar sistema preparado
with open('sistema_preparado.pdb', 'w') as f:
    app.PDBFile.writeFile(modeller.topology, modeller.positions, f)
"""

try:
    import openmm as mm
    import openmm.app as app
    import openmm.unit as unit

    pdb = app.PDBFile("estructuras_dm/1AKI_limpio.pdb")
    forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
    modeller = app.Modeller(pdb.topology, pdb.positions)
    modeller.addHydrogens(forcefield, pH=7.0)
    modeller.addSolvent(
        forcefield,
        model='tip3p',
        padding=1.0*unit.nanometer,
        ionicStrength=0.15*unit.molar
    )
    print(f"✓ Sistema preparado con OpenMM")
    print(f"  Átomos totales: {modeller.topology.getNumAtoms():,}")
    print(f"  Residuos totales: {modeller.topology.getNumResidues():,}")
    with open('estructuras_dm/1AKI_openmm.pdb', 'w') as f:
        app.PDBFile.writeFile(modeller.topology, modeller.positions, f)

except ImportError:
    print("OpenMM no disponible. Código de referencia:")
    print(codigo_openmm)
except Exception as e:
    print(f"Error: {e}")
    print("Código de referencia:")
    print(codigo_openmm)

## 8. Ejercicios

### Ejercicio 1 (Básico)
Descarga la estructura de la ubiquitina humana (PDB: 1UBQ) y límpiala usando `limpiar_pdb`. Cuenta el número de residuos y estima el número de moléculas de agua necesarias para una caja dodecaedro con d_borde = 1.2 nm.

### Ejercicio 2 (Intermedio)
Modifica el script de GROMACS para preparar una membrana lipídica DPPC usando `insane.py` o `CHARMM-GUI`. Explica las diferencias en el protocolo de preparación comparado con una proteína globular en solución.

### Ejercicio 3 (Avanzado)
Usando OpenMM, prepara y minimiza el sistema de la lisozima con el campo de fuerza AMBER14. Reporta: número de átomos total, dimensiones de la caja, número de iones añadidos, y energía potencial antes y después de la minimización.

In [ ]:
# Ejercicio 1
pdb_1ubq = descargar_pdb("1UBQ")
if pdb_1ubq:
    parser = PDB.PDBParser(QUIET=True)
    struct = parser.get_structure("1ubq", pdb_1ubq)
    
    n_residuos = sum(
        1 for res in struct.get_residues()
        if res.id[0] == ' '  # solo aminoácidos
    )
    print(f"1UBQ - Residuos de aminoácidos: {n_residuos}")
    
    r = estimar_agua_solvatacion(n_residuos, d_borde=1.2, tipo_caja='dodecaedro')
    print(f"Radio estimado de la proteína: {r['r_proteina_nm']:.2f} nm")
    print(f"Dimensión de la caja:          {r['L_caja_nm']:.2f} nm")
    print(f"Volumen de la caja:            {r['V_caja_nm3']:.1f} nm³")
    print(f"Moléculas de agua estimadas:   {r['n_agua']:,}")
    print(f"Átomos totales estimados:      {r['n_atomos_total']:,}")

## 9. Recursos Adicionales

- **Tutoriales GROMACS:**
  - [Lysozyme in Water Tutorial](http://www.mdtutorials.com/gmx/lysozyme/) — Tutorial clásico de DM con GROMACS
  - [GROMACS pdb2gmx](https://manual.gromacs.org/documentation/current/onlinehelp/gmx-pdb2gmx.html)
  - [GROMACS solvate](https://manual.gromacs.org/documentation/current/onlinehelp/gmx-solvate.html)

- **CHARMM-GUI:**
  - [CHARMM-GUI Solution Builder](https://www.charmm-gui.org/?doc=input/solution)
  - [CHARMM-GUI Membrane Builder](https://www.charmm-gui.org/?doc=input/membrane)

- **Campos de fuerza:**
  - [AMBER Force Fields](https://ambermd.org/AmberModels.php)
  - [CHARMM Force Fields](https://www.charmm.org/charmm/resources/charmm-force-fields/)
  - [GROMOS Force Fields](https://www.gromos.net/)

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Limpiar estructuras PDB para eliminar moléculas de agua y heteroátomos
- ✅ Añadir hidrógenos y asignar estados de protonación con GROMACS y OpenMM
- ✅ Solvatatar un sistema con un modelo de agua explícita (TIP3P, SPC/E)
- ✅ Neutralizar el sistema y ajustar la concentración salina
- ✅ Construir cajas de simulación con las dimensiones correctas

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 5.5: Preparación de Sistemas**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_5.4-Termostatos_y_Barostatos-blue.svg)](04_termostatos_barostatos.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_5.6_➡️-Simulación_de_Proteínas-green.svg)](06_simulacion_proteinas.ipynb)

---

📚 **[Volver al Módulo 5](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>